# A keycard protocol: from the π-calculus to the reflective ρ-calculus

This notebook models a hotel keycard protocol — guest checks in, is issued a key,
later redeems it — translating it from the polyadic π-calculus into Stratum's
reflective ρ-calculus, and explores its behaviour with the toolkit.

Pick the **Stratum** kernel to run it.

References: Meredith & Radestock, *A Reflective Higher-order Calculus* (ENTCS
141(5), 2005); Stian Lybech, *Encodability and Separation for a Reflective
Higher-Order Calculus* (EXPRESS/SOS 2022).

## 1. Generating all traces

`#explore` builds the complete trace LTS — every reachable state and every
interleaving of communications, up to a state bound. A *trace* is a path through
it. There is no directive that lists every maximal path (the number is generally
exponential), so we script a depth-first enumeration with `#rune`.

A minimal branching example: two independent communications form a *diamond* with
two interleavings.

In [ ]:
#define diamond
new a, b

a!(0) | b!(0) | a(x).0 | b(y).0

In [ ]:
#explore diamond -> dg

In [ ]:
#rune
use stratum::*;

let lts = explore(get("diamond"), 1000);
if lts.is_truncated() { println("truncated — raise the bound"); }

// DFS: every maximal path from the initial state, cutting a branch that would
// revisit a state already on the current path (cycle guard).
let traces = [];
let stack = [(lts.initial(), [lts.initial()])];
while !stack.is_empty() {
    let top = stack.pop().unwrap();
    let s = top.0;
    let path = top.1;
    let extended = false;
    for t in lts.successors(s) {
        if !path.iter().any(|u| u == t) {
            let np = path.clone();
            np.push(t);
            stack.push((t, np));
            extended = true;
        }
    }
    if !extended { traces.push(path); }
}

println(`${traces.len()} maximal trace(s):`);
for tr in traces {
    let s = "";
    for x in tr { s = `${s} s${x}`; }
    println(`  [${s} ]`);
}

## 2. The protocol in the π-calculus

$$\mathsf{Guest}(c) \;=\; \nu r.\; \overline{desk}\langle c, r\rangle.\; r(k).\; [\mathsf{Away}].\; \overline{desk}\langle \mathsf{redeem}, k, r\rangle.\; r(c'').\; \mathbf{0}$$

$$\mathsf{Att} \;=\; {!}\,\big(\, desk(c, t).\; \nu k.\; \overline{t}\langle k\rangle.\; \mathsf{Hook}(k, c) \,\big)$$

$$\mathsf{Hook}(k, c) \;=\; desk(m, k', r').\; [\,m {=} \mathsf{redeem}\,][\,k' {=} k\,]\; \overline{r'}\langle c\rangle.\; \mathbf{0}$$

Three features of this term have **no direct ρ-calculus surface form**:

- **polyadic I/O** — `desk⟨c,r⟩`, `desk⟨redeem,k,r⟩` — ρ is monadic (one message per action);
- **name-match guards** `[m=redeem][k'=k]` and the literal `redeem` — ρ has neither match nor data literals;
- the **`[Away]` box** — not a process construct at all.

The rest of the notebook shows how each is handled — or why it provably cannot be.

## 3. A capability-based translation

The key ρ idiom: **a fresh name *is* an unforgeable capability.** So `[k'=k]`
becomes *channel identity* — the Hook listens *on* the key channel `k`, rather
than comparing keys — which deletes both the `redeem` tag and both match guards.
Polyadic messages become an attendant-driven ask/answer over a private session
channel. `[Away]` is a placeholder internal step here (revisited in §4).

Note: identifiers must be globally distinct — reusing one as both a `new` name
and an input binder causes capture. `desk!(*gRep)` is the paper's name-output
`x⟨y⟩ ≜ x⟨|⌝y⌜|⟩`.

In [ ]:
#define hotel
def desk { @0 }

def Att {
  new attAsk, attKey
  ( desk(attSess).
      ( attSess!(*attAsk)
      | attAsk(attCred).
          ( attSess!(*attKey)
          | attKey(attRep). attRep!(*attCred) ) ) )
}

def Guest(gCred) {
  new gRep, gAway
  ( desk!(*gRep)
  | gRep(gAsk). ( gAsk!(*gCred)
                | gRep(gKey). ( gAway!(0) | gAway(gIgn).
                    ( gKey!(*gRep) | gRep(gConf). 0 ) ) ) )
}

Att | Guest(@(@0!(0)))

In [ ]:
#explore hotel -> hg

The trace runs to a single terminal `0`: the whole session — check-in, key
issuance, redemption on the key channel, confirmation — completes with no
deadlock and no residue.

## 4. The `[Away]` black box: returns, or doesn't

"The guest returns *or* abandons" is a nondeterministic **internal choice**. ρ
has no `+` operator, so we obtain choice from a **race**: one `gCoin!(0)` message
with two competing receivers — one redeems, the other stays away forever.

In [ ]:
#define away_sys
def desk { @0 }

def Att {
  new attAsk, attKey
  ( desk(attSess).
      ( attSess!(*attAsk)
      | attAsk(attCred).
          ( attSess!(*attKey)
          | attKey(attRep). attRep!(*attCred) ) ) )
}

def Guest(gCred) {
  new gRep, gCoin
  ( desk!(*gRep)
  | gRep(gAsk). ( gAsk!(*gCred)
                | gRep(gKey).
                    ( gCoin!(0)
                    | gCoin(rA). ( gKey!(*gRep) | gRep(gConf). 0 )
                    | gCoin(rB). 0 ) ) ) }

Att | Guest(@(@0!(0)))

In [ ]:
#explore away_sys -> ag

The LTS now **branches** (the coin race), reaching two outcomes:

- **returns** → the guest redeems and the protocol completes;
- **stays away** → the attendant's Hook is left **blocked forever** on the key
  channel — an un-redeemed key, faithfully modelled as a stuck residual process.

Because there is no `+`, the un-chosen branch leaves a dangling "loser" residue,
so this is behaviourally *adequate* but not fully faithful. Distinguishing
"abandoned" from "merely delayed" is a fairness / epistemic question — checked at
the logic layer (`FairAf`, `K_A`/`P_A`), not encoded in the process. That is what
the `?` and observation set in `[Away]_{\{desk,r,k\}}^{?}` denote.

## 5. Fresh keys under replication: a runtime name allocator

Stratum's `new` is *static* (one ground name per parse), so a replicated
attendant would hand every guest the *same* key. Lybech's encoding of π's `ν`
into ρ generates names **at runtime**. Ported here as a reusable allocator: a
state cell `st` holds the current name and returns a fresh one per request,
incrementing via dynamic quoting (`lift`): `+x = @(x(z).0)`.

Allocated names are **input-shaped** `@(x(z).0)`, structurally disjoint from the
lift-shaped `new` ground names `@(@0!(…))` — the disjointness condition whose
violation invalidated Meredith's original encoding (Lybech §4). Each request
`alloc!(*rep) | rep(x)…` is exactly Lybech's `⟦(νx)P⟧ = p(x).⟦P⟧ | p⟨n⟩`.

In [ ]:
#define alloc_demo
def allocServer { st(cur). alloc(rep). ( rep!(*cur) | st!(cur(z).0) ) }

new st, alloc, r1, r2, seed
( st!(seed(z).0)
| allocServer | allocServer
| ( alloc!(*r1) | r1(nm1). nm1!(0) )
| ( alloc!(*r2) | r2(nm2). nm2!(0) ) )

In [ ]:
#explore alloc_demo -> alg

A single terminal (the allocator is confluent), and the two clients' marks
`nm!(0)` land on **distinct, disjoint** channels — distinct fresh names.

### The replicated attendant

Two attendant instances share one allocator; each draws a fresh key per guest,
so the two guests are guaranteed **distinct** keys. The `#rune` cell explores the
system and prints each terminal — you can read off the two different key channels.

In [ ]:
#define hotel_fresh
def allocServer { st(cur). alloc(rep). ( rep!(*cur) | st!(cur(z).0) ) }

new st, alloc, seed
new desk
new g1rep, g1krep, g2rep, g2krep
( st!(seed(z).0)
| allocServer | allocServer
| desk(a1). ( alloc!(*g1krep) | g1krep(fk). a1!(*fk) )
| desk(a2). ( alloc!(*g2krep) | g2krep(fk). a2!(*fk) )
| ( desk!(*g1rep) | g1rep(k1). k1!(0) )
| ( desk!(*g2rep) | g2rep(k2). k2!(0) ) )

In [ ]:
#rune
use stratum::*;

let lts = explore(get("hotel_fresh"), 4000);
println(`states: ${lts.num_states()}, transitions: ${lts.num_transitions()}, truncated: ${lts.is_truncated()}`);
for i in 0..lts.num_states() {
    if lts.successors(i).is_empty() {
        println(`terminal s${i}: ${lts.state(i).source()}`);
    }
}

## Where the model stops

- **Bounded replication** (a fixed number of attendant instances, as above) is
  faithful and model-checkable.
- **Unbounded** `!`-replication has two obstacles in Stratum today: `bang` /
  `contract` unfold *eagerly*, so the LTS truncates (Lybech's lazy,
  input-guarded replication `!u(v).P` would keep finite runs finite); and each
  instance would need to draw its *own* reply channel from the allocator (the
  full name-parameter threading).
- **Choice** (`[Away]`) and **data / match** lie outside the choice-free
  asynchronous fragment that the π→ρ encoding covers. And ρ is strictly more
  expressive in the other direction: there is no encoding of ρ into π (Lybech,
  Theorem 1).